In [1]:
import re
import math

In [2]:
import pandas as pd
df = pd.read_csv("bbc-text (1).csv", encoding = 'latin1')
print(df.head())

        category                                               text
0           tech  tv future in the hands of viewers with home th...
1       business  worldcom boss  left books alone  former worldc...
2          sport  tigers wary of farrell  gamble  leicester say ...
3          sport  yeading face newcastle in fa cup premiership s...
4  entertainment  ocean s twelve raids box office ocean s twelve...


# 1.	Data Preprocessing

a.	Clean the text data by removing punctuation, converting to lowercase, and removing stop words.

b.	Tokenize the text into individual words.


In [3]:
df['text'] = df['text'].str.lower()

In [4]:
df['text'] = df['text'].str.replace(r'[^a-zA-Z\d\s]', '', regex = True)

In [5]:
df['text'] = df['text'].str.split()

In [6]:
print(df['text'].head())

0    [tv, future, in, the, hands, of, viewers, with...
1    [worldcom, boss, left, books, alone, former, w...
2    [tigers, wary, of, farrell, gamble, leicester,...
3    [yeading, face, newcastle, in, fa, cup, premie...
4    [ocean, s, twelve, raids, box, office, ocean, ...
Name: text, dtype: object


In [7]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

def remove_stopwords(words_list):
  filtered_words = []
  for w in words_list:
    if w not in stop_words:
      filtered_words.append(w)
  return filtered_words

df['text'] = df['text'].apply(remove_stopwords)

print(df['text'].head())

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


0    [tv, future, hands, viewers, home, theatre, sy...
1    [worldcom, boss, left, books, alone, former, w...
2    [tigers, wary, farrell, gamble, leicester, say...
3    [yeading, face, newcastle, fa, cup, premiershi...
4    [ocean, twelve, raids, box, office, ocean, twe...
Name: text, dtype: object


# 2.	Implement Bag of Words

a.	Create a function to build a vocabulary from the training set.


In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df['text'],
    df['category'],
    test_size = 0.2,
    random_state = 42
)

In [9]:
def build_vocabulary(text_column):
  vocab = set()
  for text in text_column:
    for word in text:
      vocab.add(word)

  return list(vocab)

vocabulary = build_vocabulary(X_train)

print("Total number of words in the vocabulary:", len(vocabulary))

Total number of words in the vocabulary: 29291


b.	Implement a function that converts a document into a BoW vector.

In [10]:
def text_to_vector(text, vocabulary):
  vector = []
  for word in vocabulary:
    count = text.count(word)
    vector.append(count)

  return vector

c.	Use your implementation to represent each document in both training and testing sets as a BoW vector.

In [11]:
X_train_vectors = []
for text in X_train:
  vector = text_to_vector(text, vocabulary)
  X_train_vectors.append(vector)

X_test_vectors = []
for text in X_test:
  vector = text_to_vector(text, vocabulary)
  X_test_vectors.append(vector)

# 3.	Implement TF-IDF:
a.	Create a function to calculate term frequency (TF) for each term in a document.

In [12]:
def calculate_tf(text, vocabulary):
  vector = []
  text_length = len(text)

  if text_length == 0:
    return [0] * len(vocabulary)

  for word in vocabulary:
    count = text.count(word)
    tf_value = count / text_length
    vector.append(tf_value)

  return vector

b.	Implement a function to calculate inverse document frequency (IDF) for each term in the corpus.

In [13]:
def calculate_idf(X_train, vocabulary):
  N = len(X_train)
  idf_vector = []

  for word in vocabulary:

    df_count = 0
    for text in X_train:
      if word in text:
        df_count += 1

    idf_value = math.log(N / df_count + 1)
    idf_vector.append(idf_value)

  return idf_vector

c.	Combine TF and IDF to create TF-IDF vectors for each document.

In [14]:
idf_vector = calculate_idf(X_train, vocabulary)

X_train_tfidf = []

for text in X_train:
  tf_vector = calculate_tf(text, vocabulary)

  tfidf_vector = []
  for tf, idf in zip(tf_vector, idf_vector):
    tfidf_vector.append(tf * idf)

  X_train_tfidf.append(tfidf_vector)

print("Number of articles:", len(X_train_tfidf))

Number of articles: 1780


In [15]:
X_test_tfidf = []

for text in X_test:
    tf_vector = calculate_tf(text, vocabulary)

    tfidf_vector = []
    for tf, idf in zip(tf_vector, idf_vector):
      tfidf_vector.append(tf * idf)

    X_test_tfidf.append(tfidf_vector)

print("Number of articles:", len(X_test_tfidf))

Number of articles: 445


In [16]:
print("Vocabulary size:", len(vocabulary))
print("Length of the first vector in X_train:", len(X_train_tfidf[0]))
print("Length of the first vector in X_test:", len(X_test_tfidf[0]))
print('--------------------------------------------')
print("Original X_train count:", len(X_train))
print("Vectorized X_train_tfidf count:", len(X_train_tfidf))
print('--------------------------------------------')
print("Original X_test count:", len(X_test))
print("Vectorized X_test_tfidf count:", len(X_test_tfidf))

Vocabulary size: 29291
Length of the first vector in X_train: 29291
Length of the first vector in X_test: 29291
--------------------------------------------
Original X_train count: 1780
Vectorized X_train_tfidf count: 1780
--------------------------------------------
Original X_test count: 445
Vectorized X_test_tfidf count: 445


# 4.	Analysis
a.	For a given category, find the top 10 words with the highest average TF-IDF scores.

In [17]:
target_category = "business"

category_vectors = []

for vector, label in zip(X_train_tfidf, y_train):
  if label == target_category:
    category_vectors.append(vector)

num_features = len(vocabulary)
ave_tfidf = [0] * num_features

for i in range(num_features):
  word_scores = []
  for vector in category_vectors:
    word_scores.append(vector[i])
  if len(category_vectors) > 0:
    ave_tfidf[i] = sum(word_scores) / len(category_vectors)

word_score_pairs = []
for word, score in zip(vocabulary, ave_tfidf):
    word_score_pairs.append((word, score))

word_score_pairs.sort(key=lambda x: x[1], reverse=True)

print(f"Top 10 words for category '{target_category}':")
for word, score in word_score_pairs[:10]:
  print(f"- {word}: {score:.4f}")

Top 10 words for category 'business':
- said: 0.0128
- us: 0.0100
- growth: 0.0095
- bank: 0.0090
- oil: 0.0090
- market: 0.0090
- shares: 0.0090
- company: 0.0089
- economy: 0.0089
- firm: 0.0081


b.	Identify words that have high TF scores but low IDF scores, and vice versa.

In [21]:
num_features = len(vocabulary)
ave_tf = [0] * num_features

for text in X_train:
  tf_v = calculate_tf(text, vocabulary)
  for i in range(num_features):
    ave_tf[i] += tf_v[i]

for i in range(num_features):
  ave_tf[i] = ave_tf[i] / len(X_train)

word_tf_idf_pairs = []
for word, tf_val, idf_val in zip(vocabulary, ave_tf, idf_vector):
    word_tf_idf_pairs.append((word, tf_val, idf_val))


word_tf_idf_pairs.sort(key=lambda x: x[1], reverse=True)

print("Words with High TF and Low IDF:")
for word, tf_val, idf_val in word_tf_idf_pairs[:5]:
  print(f"- Word: {word}, Mean TF: {tf_val:.4f}, IDF: {idf_val:.4f}")

print("\n" + "="*40 + "\n")

word_tf_idf_pairs.sort(key=lambda x: x[1])

print("Words with Low TF and High IDF:")
for word, tf_val, idf_val in word_tf_idf_pairs[:5]:
  print(f"- Word: {word}, Mean TF: {tf_val:.2e}, IDF: {idf_val:.4f}")

Words with High TF and Low IDF:
- Word: said, Mean TF: 0.0153, IDF: 0.7781
- Word: mr, Mean TF: 0.0055, IDF: 1.3464
- Word: would, Mean TF: 0.0049, IDF: 1.0745
- Word: also, Mean TF: 0.0045, IDF: 1.0123
- Word: year, Mean TF: 0.0041, IDF: 1.1735


Words with Low TF and High IDF:
- Word: bane, Mean TF: 3.08e-07, IDF: 7.4849
- Word: pleeease, Mean TF: 3.08e-07, IDF: 7.4849
- Word: baffling, Mean TF: 3.08e-07, IDF: 7.4849
- Word: redifining, Mean TF: 3.08e-07, IDF: 7.4849
- Word: matronic, Mean TF: 3.08e-07, IDF: 7.4849
